In [45]:
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Load Documents

In [46]:
DATASET_PATH = "../Dataset"

def load_documents():
    docs = {}
    doc_dir = os.path.join(DATASET_PATH, "documents")
    
    for file in os.listdir(doc_dir):
        if file.endswith(".txt"):
            name = file.replace(".txt", "")
            file_path = os.path.join(doc_dir, file)
            with open(file_path, "r", encoding="utf-8") as f:
                docs[name] = f.read()
    return docs

def load_all_qa():
    all_qa = []
    qa_dir = os.path.join(DATASET_PATH, "qa")
    
    for file in os.listdir(qa_dir):
        if file.endswith(".json"):
            file_path = os.path.join(qa_dir, file)
            with open(file_path, "r", encoding="utf-8") as f:
                all_qa.extend(json.load(f))
    return all_qa

Load LLM models

In [47]:
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16,
        load_in_4bit=True,
        llm_int8_enable_fp32_cpu_offload=True
    )

    model.eval()
    return tokenizer, model

In [48]:
MISTRAL = "mistralai/Mistral-7B-Instruct-v0.2"
QWEN = "Qwen/Qwen2-7B-Instruct"

Metric Utils

In [49]:
# Token Counter
def count_tokens(tokenizer, text):
    return len(tokenizer(text)["input_ids"])

In [50]:
# Exact Match
def exact_match(pred, gold):
    return int(pred.strip() == gold.strip())

In [51]:
# Timed Generation
def generate(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.0
        )
    end = time.time()

    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    latency = end - start
    input_tokens = inputs["input_ids"].shape[1]
    output_tokens = outputs.shape[1] - input_tokens

    return output_text, latency, input_tokens + output_tokens

LLM Systems

Prompt Only

In [52]:
def run_prompt_only(model, tokenizer, docs, qa_data):
    results = []

    for item in qa_data:
        doc_text = docs[item["document"]]

        prompt = f"""
Extract the exact answer span from the document.

Document:
{doc_text}

Question:
{item["question"]}

Answer:
"""

        output, latency, tokens = generate(model, tokenizer, prompt)

        results.append({
            "qa_id": item["id"],
            "system": "Prompt-Only",
            "tokens": tokens,
            "latency": latency,
            "EM": exact_match(output, item["answer_text"])
        })

    return results

RAG

In [53]:
def chunk_text(tokenizer, text, size=512, overlap=128):
    tokens = tokenizer(text)["input_ids"]
    chunks = []

    for i in range(0, len(tokens), size - overlap):
        chunk_tokens = tokens[i:i+size]
        chunks.append(tokenizer.decode(chunk_tokens))

    return chunks

In [54]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def build_index(docs, tokenizer):
    all_chunks = []
    metadata = []

    for name, text in docs.items():
        chunks = chunk_text(tokenizer, text)
        for chunk in chunks:
            all_chunks.append(chunk)
            metadata.append(name)

    embeddings = embedder.encode(all_chunks, convert_to_numpy=True)

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)

    return index, all_chunks

In [55]:
def run_rag(model, tokenizer, index, chunks, qa_data):
    results = []

    for item in qa_data:
        q_emb = embedder.encode(item["question"], convert_to_numpy=True).reshape(1, -1)

        start_retrieval = time.time()
        D, I = index.search(q_emb, 5)
        retrieval_time = time.time() - start_retrieval

        retrieved_chunks = [chunks[i] for i in I[0]]
        context = "\n\n".join(retrieved_chunks)

        prompt = f"""
Extract exact answer span.

Context:
{context}

Question:
{item["question"]}

Answer:
"""

        output, gen_time, tokens = generate(model, tokenizer, prompt)

        recall = int(any(item["answer_text"] in c for c in retrieved_chunks))

        results.append({
            "qa_id": item["id"],
            "system": "RAG",
            "tokens": tokens,
            "latency": retrieval_time + gen_time,
            "EM": exact_match(output, item["answer_text"]),
            "recall5": recall
        })

    return results

Context Optimization

In [56]:
def run_context(model, tokenizer, docs, qa_data):
    results = []

    for item in qa_data:
        doc_text = docs[item["document"]]

        article = item["article"]
        context = ""

        for line in doc_text.split("\n"):
            if article in line:
                context += line + "\n"

        if context == "":
            context = doc_text[:3000]

        prompt = f"""
Extract exact answer span.

Context:
{context}

Question:
{item["question"]}

Answer:
"""

        output, latency, tokens = generate(model, tokenizer, prompt)

        results.append({
            "qa_id": item["id"],
            "system": "Context",
            "tokens": tokens,
            "latency": latency,
            "EM": exact_match(output, item["answer_text"])
        })

    return results

Run

In [59]:
docs = load_documents()
qa_data = load_all_qa()

# Run for Mistral
tok_m, model_m = load_model(MISTRAL)

index, chunks = build_index(docs, tok_m)

results = []
results += run_prompt_only(model_m, tok_m, docs, qa_data)
results += run_rag(model_m, tok_m, index, chunks, qa_data)
results += run_context(model_m, tok_m, docs, qa_data)

# Repeat for Qwen
tok_q, model_q = load_model(QWEN)

results += run_prompt_only(model_q, tok_q, docs, qa_data)
results += run_rag(model_q, tok_q, index, chunks, qa_data)
results += run_context(model_q, tok_q, docs, qa_data)

with open("final_results.json", "w") as f:
    json.dump(results, f, indent=2)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.23 GiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 9.35 GiB is allocated by PyTorch, and 1.18 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Plot and Aggregrate

In [ ]:
df = pd.read_json("final_results.json")

summary = df.groupby(["system"]).agg({
    "tokens": "mean",
    "latency": "mean",
    "EM": "mean",
    "recall5": "mean"
})

print(summary)

In [ ]:
# Token usage plot
summary["tokens"].plot(kind="bar")
plt.ylabel("Average Tokens")
plt.title("Average Token Usage per System")
plt.show()

In [ ]:
# Latency Plot

summary["latency"].plot(kind="bar")
plt.ylabel("Average Latency (s)")
plt.title("Average Latency per System")
plt.show()

In [ ]:
# Tokens vs Latency Trageoff
plt.scatter(summary["tokens"], summary["latency"])

for i, txt in enumerate(summary.index):
    plt.annotate(txt, (summary["tokens"][i], summary["latency"][i]))

plt.xlabel("Average Tokens")
plt.ylabel("Average Latency")
plt.title("Token vs Latency Tradeoff")
plt.show()

In [58]:
print(docs.keys())

dict_keys(['AIA', 'DGA', 'DMA', 'DSA', 'GDPR'])
